# Driver — độ bền của static Android malware detection dưới obfuscation

Notebook này chỉ **điều khiển**; mọi logic nằm trong `src/`. Chạy tuần tự theo phase.

| Phase | Cell | Thời gian | Chạy lại mỗi session? |
|---|---|---|---|
| 0 | Setup + smoke test | 30 phút | Có |
| 1 | Giải nén dataset + manifest + split | ~30 phút | Có (trừ manifest/split đã ở Drive) |
| 2 | Trích feature APK sạch | ~1,1 giờ | Không — checkpoint ở Drive |
| 3 | Baseline ML + eval sạch | ~1 giờ | Không |
| 4 | Obfuscation | 9–10 giờ, **chia 2 session** | Resume từ `obf_progress.json` |
| 5 | Ma trận kết quả | ~2 giờ | Không |

**Đừng đi tiếp nếu smoke test ở Phase 0 hỏng.**

Hai việc phải làm thủ công một lần, vì không tự động hoá được:

- **Dataset** nằm sau form đăng ký của CIC — xem Phase 1.
- **Obfuscapk** không có trên PyPI — cell ngay dưới lo việc này.

## Phase 0 — Setup

In [ ]:
# Mount Drive để checkpoint
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/apk-robustness'
SCRATCH = '/content/apkrob'

import os
os.environ['APKROB_WORK'] = WORK
os.environ['APKROB_SCRATCH'] = SCRATCH
os.makedirs(WORK, exist_ok=True)
print('WORK   ', WORK)
print('SCRATCH', SCRATCH)

In [ ]:
# Java + Android build-tools cho Obfuscapk.
# apktool KHÔNG có trong danh sách gốc của PLAN nhưng Obfuscapk bắt buộc phải có —
# thiếu nó thì mọi kỹ thuật đều fail ngay ở bước decompile.
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jdk-headless apktool zipalign apksigner

### Obfuscapk — ba cái bẫy, không phải một

PLAN mục 2 ghi `pip install obfuscapk`. Lệnh đó không bao giờ chạy được, và vòng qua được nó rồi thì còn hai cái nữa đợi sẵn.

**1. Không có trên PyPI.** `pip install obfuscapk` trả `No matching distribution found`. Repo cũng không có `setup.py` ở gốc nên `pip install git+...` cũng hỏng. Cách duy nhất: clone rồi đưa `src/` của nó vào `PYTHONPATH` qua biến `OBFUSCAPK_SRC`.

**2. Yapsy hỏng trên Python 3.12 trở lên.** `src/requirements.txt` của Obfuscapk ghim `Yapsy==1.12.2` — bản mới nhất trên PyPI, phát hành 2019 — và bản đó `import imp`, module đã bị xoá khỏi Python 3.12. Colab hiện chạy **Python 3.13**. Nhánh master của Yapsy đã chuyển sang `importlib` nhưng chưa bao giờ được phát hành, nên phải cài từ git.

**3. BundleDecompiler.** `check_external_tool_dependencies()` khởi tạo BundleDecompiler **vô điều kiện**, dù README gọi nó là tuỳ chọn. Thiếu biến môi trường thì constructor ném `TypeError: stat: path should be string... not NoneType` — một bug của chính Obfuscapk, và vì nó chạy trước cả argparse nên ngay `--help` cũng chết.

`src/obfuscate.py` tự lo cái thứ 3: constructor chỉ gọi `os.path.isfile` chứ không chạy jar, và pipeline này chỉ truyền `.apk` chứ không bao giờ truyền `.aab`, nên một file người thế có `chmod +x` là đủ và an toàn. Nếu sau này cần xử lý `.aab` thật thì tải jar thật rồi đặt `BUNDLE_DECOMPILER_PATH` — code không ghi đè biến bạn đã đặt.

Các pin còn lại của Obfuscapk cũng từ 2021 (`pycryptodome==3.12.0`), không có wheel cho Python 3.13, nên cell dưới cài bản mới nhất thay vì theo pin.

Cell "Kiểm tra toolchain" bên dưới báo chính xác cái nào hỏng, nếu có.

In [ ]:
import os, sys
print('Python', sys.version.split()[0])

# 1. Obfuscapk: clone, KHÔNG pip install (không có trên PyPI)
OBF = '/content/Obfuscapk'
if not os.path.isdir(OBF):
    !git clone -q --depth 1 https://github.com/ClaudiuGeorgiu/Obfuscapk.git {OBF}
os.environ['OBFUSCAPK_SRC'] = f'{OBF}/src'   # src/obfuscate.py đọc biến này

# 2. Phụ thuộc của Obfuscapk — bỏ pin cũ, trừ Yapsy phải lấy từ git
#    (bản trên PyPI dùng 'imp', đã bị xoá khỏi Python 3.12+)
!pip -q install pycryptodome tqdm vt-py
!pip -q install 'yapsy @ git+https://github.com/tibonihoo/yapsy.git@master#subdirectory=package'

# 3. Phụ thuộc của dự án này
!pip -q install androguard==4.1.2 scikit-learn pandas pyarrow xgboost networkx joblib

print('OBFUSCAPK_SRC =', os.environ['OBFUSCAPK_SRC'])

In [ ]:
# Lấy code của dự án này.
REPO = 'https://github.com/trantrien1/AndroiDetection1.git'
PROJ = '/content/AndroiDetection1'

import os
if not os.path.isdir(PROJ):
    !git clone -q {REPO} {PROJ}
else:
    !git -C {PROJ} pull -q          # lấy bản mới nhất khi chạy lại session

%cd {PROJ}
if not os.path.isdir('src'):
    raise SystemExit(f'Chưa có code trong {PROJ}/src — kiểm tra lại REPO.')
print('OK:', sorted(os.listdir('src')))

In [ ]:
# Kiểm tra toolchain TRƯỚC khi tải dataset — apktool, apksigner, zipalign và
# bản thân Obfuscapk. In ra chính xác cái nào hỏng và cách sửa.
# Phải in "THIEU / HONG: khong co - san sang" thì mới đi tiếp.
!python -m src.obfuscate --check

## Phase 1 — Lấy dataset + manifest + chốt split

**URL trong PLAN mục 3 đã chết.** `cicresearch.ca/.../Dataset/APKs/` giờ 302 về trang giới thiệu datasets, và CIC đã đặt toàn bộ dataset sau một **form đăng ký** (họ tên, email, tổ chức, chức danh, quốc gia). Không còn đường tải ẩn danh, nên notebook không thể tự tải hộ.

Tải về một lần, để trên Drive, rồi dùng lại mãi:

1. Mở [trang dataset](https://www.unb.ca/cic/datasets/maldroid-2020.html) → **Download the dataset** → điền form.
2. Tải 5 file zip theo category vào một thư mục trên Drive, ví dụ `MyDrive/maldroid_zips/`.
3. Chỉnh `ZIPS` ở cell dưới cho trỏ đúng thư mục đó.

Tên file không cần khớp chính xác — `find_local_zip` khớp không phân biệt hoa thường, miễn tên file có chứa tên category (`Benign`, `Adware`, `Banking`, `SMS`, `Riskware`).

Đặt zip trên Drive chứ đừng đặt ở `/content`: `/content` bị xoá mỗi session, còn APK giải nén thì vẫn nên để ở `/content` cho nhanh.

In [ ]:
# Trỏ tới thư mục chứa 5 file zip đã tải thủ công về Drive.
ZIPS = '/content/drive/MyDrive/maldroid_zips'

import os
if os.path.isdir(ZIPS):
    print(sorted(f for f in os.listdir(ZIPS) if f.lower().endswith('.zip')))
else:
    print('CHUA CO:', ZIPS, '- xem hướng dẫn ở cell trên')

In [ ]:
# Giải nén + sinh manifest.csv. Chạy lại được: category nào đã giải nén thì bỏ qua.
!python -m src.download --zip-dir "{ZIPS}"

# Nếu bạn đã tự giải nén sẵn thành apks/<category>/*.apk thì dùng dòng này:
# !python -m src.download --manifest-only

In [ ]:
# Chốt split. Sau lần chạy đầu, test_sha256.txt KHÔNG BAO GIỜ đổi nữa:
# các lần chạy sau sẽ tự đọc lại file đã chốt.
!python -m src.split

In [ ]:
# SMOKE TEST — PLAN mục 2: chạy Obfuscapk trên 1 APK với Rebuild.
# Nếu cell này fail thì DỪNG LẠI, mọi thứ sau đều vô nghĩa.
!python -m src.obfuscate --smoke-test

## Phase 2 — Trích feature tĩnh

Chỉ trích 7.500 APK trong split đã chốt, không phải cả 17k. Checkpoint mỗi 500 APK xuống Drive nên cell này resume được sau khi session chết.

In [ ]:
# Thử 20 APK trước để biết tốc độ thực tế và tỉ lệ fail
!python -m src.features.extract --tag clean --limit 20

In [ ]:
!python -m src.features.extract --tag clean
!cat $APKROB_WORK/features/extract_stats_clean.json

## Phase 3 — Baseline ML + eval sạch

In [ ]:
# Train model toàn bộ feature (rf/xgb/svm) + model chỉ-một-nhóm cho Bảng B
!python -m src.train

In [ ]:
!python -m src.evaluate --val --clean

In [ ]:
# Baseline đối chiếu trên CSV gốc của CIC — con số này so được trực tiếp với
# literature. Tải CSV từ trang CICMalDroid rồi trỏ đường dẫn vào đây.
CSV = '/content/drive/MyDrive/apk-robustness/feature_vectors_syscallsbinders_frequency_5_Cat.csv'
!test -f "$CSV" && python -m src.train --csv-baseline "$CSV" || echo 'Chưa có CSV — bỏ qua bước đối chiếu'

## Phase 4 — Obfuscation (9–10 giờ, chia 2 session)

Session A chạy T1–T3, session B chạy T4–T6. `obf_progress.json` nằm trên Drive nên chạy lại cell là resume, không làm lại từ đầu.

In [ ]:
# SESSION A
!python -m src.obfuscate --techniques T1_trivial T2_rename T3_string --workers 4

In [ ]:
# SESSION B
!python -m src.obfuscate --techniques T4_asset T5_cfg T6_reflection --workers 4

In [ ]:
!python -m src.obfuscate --report   # tỉ lệ APK hỏng theo từng kỹ thuật

In [ ]:
# Trích feature trên APK đã obfuscate.
# APK obfuscated nằm ở SCRATCH nên phải chạy cell này TRONG CÙNG session
# với cell obfuscate ở trên, trước khi Colab xoá disk.
for tech in ['T1_trivial','T2_rename','T3_string','T4_asset','T5_cfg','T6_reflection']:
    !python -m src.features.extract --tag {tech}

## Phase 5 — Ma trận kết quả

In [ ]:
!python -m src.evaluate --obf

In [ ]:
!python -m src.matrix

In [ ]:
from IPython.display import Markdown, display
import os
display(Markdown(open(os.path.join(os.environ['APKROB_WORK'], 'results', 'tables.md'), encoding='utf-8').read()))

## Pass sau — chỉ khi Bảng B đòi

`G5` (call graph) tốn 5–30s/APK, tức ~80% tổng thời gian Phase 2. Chỉ bật nếu Bảng B cho thấy G1–G4 không đủ tách bạch.

In [ ]:
# !python -m src.features.extract --tag clean --with-g5
# !python -m src.train --groups G1 G2 G3 G4 G5